Reference:

    - Perceiver IO: https://arxiv.org/abs/2107.14795

In [ ]:
# import os
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import pytorch_lightning as pl
# from torch.utils.data import DataLoader
# from typing import List, Tuple, Dict
# from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR

# # --- Make sure these imports are correct for your project structure ---
# from config import project_paths, simd_r_drive_server_config
# from models.pytorch.narrative_stack.stage2.dataset import (
#     Stage2StackDataset,
#     stage2_collate_stacks,
# )

# # ======== DIAGNOSTIC CONFIGURATION ========
# CONFIG = {
#     "lr": 5e-5, 
#     "batch_size": 8,
#     "grad_clip_val": 1.0,
#     "loss_alpha": 0.5,
#     "dropout_rate": 0.1,
#     "weight_decay": 1e-5,
#     "warmup_steps": 200,
#     "input_dim": 256,
#     "latent_dim": 256,
#     "num_latents": 64,
#     "encoder_depth": 2,
#     "query_dim": 256,
#     "decoder_depth": 2,
# }

# # ======== BUILDING BLOCKS FOR DIAGNOSTIC TEST (FINAL FIX) ========

# class PerceiverStackEncoder(nn.Module):
#     """ The encoder for a single stack. """
#     def __init__(self, input_dim: int, latent_dim: int, num_latents: int, depth: int, dropout_rate: float):
#         super().__init__()
#         self.latents = nn.Parameter(torch.randn(num_latents, latent_dim))
#         self.cross_attn = nn.MultiheadAttention(embed_dim=latent_dim, num_heads=4, batch_first=True, dropout=dropout_rate)
#         self.cross_proj = nn.Linear(input_dim, latent_dim)
#         self.query_norm = nn.LayerNorm(latent_dim)
#         self.kv_norm = nn.LayerNorm(latent_dim)
#         self.blocks = nn.ModuleList(
#             [
#                 nn.TransformerEncoderLayer(
#                     d_model=latent_dim, nhead=4, dim_feedforward=latent_dim * 4,
#                     batch_first=True, activation=F.gelu, norm_first=True,
#                     dropout=dropout_rate
#                 ) for _ in range(depth)
#             ]
#         )
#         self.output_norm = nn.LayerNorm(latent_dim)

#     def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
#         B, N, latent_dim = x.shape[0], x.shape[1], self.latents.size(-1)
#         latents = self.latents.unsqueeze(0).repeat(B, 1, 1)
#         x_proj = self.cross_proj(x)
#         latents_norm = self.query_norm(latents)
#         x_proj_norm = self.kv_norm(x_proj)

#         # The key_padding_mask is True for padded values that should be ignored.
#         key_padding_mask = ~mask

#         # --- FINAL FIX: Prevent softmax(all -inf) which results in NaN ---
#         # Identify items in the batch that are fully padded (i.e., their mask has no True values).
#         fully_padded_items = ~mask.any(dim=1)
        
#         # If any such items exist...
#         if fully_padded_items.any():
#             # ...for those specific items, set the first element of their key_padding_mask to False.
#             # This "tricks" the softmax into thinking there's one valid item to attend to,
#             # preventing the computation of softmax([-inf, -inf, ...]).
#             key_padding_mask[fully_padded_items, 0] = False

#         attn_output, _ = self.cross_attn(
#             query=latents_norm, key=x_proj_norm, value=x_proj_norm, key_padding_mask=key_padding_mask
#         )
        
#         latents = latents + attn_output
#         for block in self.blocks:
#             latents = block(latents)
#         return self.output_norm(latents).mean(dim=1)

# class PerceiverDecoder(nn.Module):
#     """ A simplified decoder for a single context vector. """
#     def __init__(self, context_dim: int, query_dim: int, output_dim: int, depth: int, dropout_rate: float):
#         super().__init__()
#         self.context_proj = nn.Linear(context_dim, query_dim)
#         self.cross_attn = nn.MultiheadAttention(embed_dim=query_dim, num_heads=4, batch_first=True, dropout=dropout_rate)
#         self.query_norm = nn.LayerNorm(query_dim)
#         self.context_norm = nn.LayerNorm(query_dim)
#         self.blocks = nn.ModuleList(
#             [
#                 nn.TransformerEncoderLayer(
#                     d_model=query_dim, nhead=4, dim_feedforward=query_dim * 4,
#                     batch_first=True, activation=F.gelu, norm_first=True,
#                     dropout=dropout_rate
#                 ) for _ in range(depth)
#             ]
#         )
#         self.output_head = nn.Linear(query_dim, output_dim)

#     def forward(self, context_vector: torch.Tensor, output_query: torch.Tensor) -> torch.Tensor:
#         projected_context = self.context_proj(context_vector)
#         latent = projected_context.unsqueeze(1).expand(-1, output_query.size(1), -1)
#         query_norm = self.query_norm(output_query)
#         latent_norm = self.context_norm(latent)
#         attn_output, _ = self.cross_attn(query=query_norm, key=latent_norm, value=latent_norm)
#         x = output_query + attn_output
#         for block in self.blocks:
#             x = block(x)
#         return self.output_head(x)


# # ======== SINGLE STACK TESTER MODULE ========
# class SingleStackTester(pl.LightningModule):
#     """A minimal autoencoder for a single stack to test if the core components can learn."""
#     def __init__(self, hparams: dict):
#         super().__init__()
#         self.save_hyperparameters(hparams)
        
#         self.encoder = PerceiverStackEncoder(
#             input_dim=self.hparams.input_dim,
#             latent_dim=self.hparams.latent_dim,
#             num_latents=self.hparams.num_latents,
#             depth=self.hparams.encoder_depth,
#             dropout_rate=self.hparams.dropout_rate
#         )
        
#         self.decoder = PerceiverDecoder(
#             context_dim=self.hparams.latent_dim,
#             query_dim=self.hparams.query_dim,
#             output_dim=self.hparams.input_dim,
#             depth=self.hparams.decoder_depth,
#             dropout_rate=self.hparams.dropout_rate
#         )

#         self.balance_embedding = nn.Embedding(num_embeddings=3, embedding_dim=self.hparams.query_dim)
#         self.period_embedding = nn.Embedding(num_embeddings=3, embedding_dim=self.hparams.query_dim)
#         self.loss_fn = nn.MSELoss()

#     def forward(self, stack, mask, balance, period):
#         latent_vector = self.encoder(stack, mask)
        
#         balance_embed = self.balance_embedding(balance)
#         period_embed = self.period_embedding(period)
#         query = balance_embed + period_embed
        
#         reconstruction = self.decoder(latent_vector, query)
#         return reconstruction

#     def training_step(self, batch, batch_idx):
#         stacks, masks, balance_batch, period_batch = batch
        
#         # We only use the first category for this test
#         stack_0, mask_0, balance_0, period_0 = stacks[0], masks[0], balance_batch[0], period_batch[0]
        
#         recon = self.forward(stack_0, mask_0, balance_0, period_0)
#         original = stack_0[mask_0]
#         recon = recon[mask_0]

#         if original.numel() == 0:
#             return None

#         mse_loss = self.loss_fn(recon, original)
#         cosine_loss = (1 - F.cosine_similarity(recon, original, dim=1, eps=1e-8)).mean()
#         loss = self.hparams.loss_alpha * mse_loss + (1 - self.hparams.loss_alpha) * cosine_loss
        
#         if not torch.isfinite(loss):
#             print(f"WARNING: Loss is NaN or Inf at batch {batch_idx}. Skipping update.")
#             return None

#         self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
#         return loss

#     def configure_optimizers(self):
#         optimizer = torch.optim.AdamW(
#             self.parameters(), 
#             lr=self.hparams.lr,
#             weight_decay=self.hparams.weight_decay
#         )
        
#         try:
#             total_steps = self.trainer.estimated_stepping_batches
#         except AttributeError:
#             total_steps = 100_000 

#         warmup_scheduler = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=self.hparams.warmup_steps)
#         main_scheduler_t_max = max(1, total_steps - self.hparams.warmup_steps)
#         main_scheduler = CosineAnnealingLR(optimizer, T_max=main_scheduler_t_max, eta_min=1e-7)

#         lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
#             optimizer, schedulers=[warmup_scheduler, main_scheduler], milestones=[self.hparams.warmup_steps]
#         )
        
#         return {
#             "optimizer": optimizer,
#             "lr_scheduler": {
#                 "scheduler": lr_scheduler,
#                 "interval": "step",
#             },
#         }

# # ======== MAIN EXECUTION ========
# def run_diagnostic():
#     print("==============================================")
#     print("     RUNNING SINGLE STACK DIAGNOSTIC TEST     ")
#     print("==============================================")
#     print("Goal: Confirm the building blocks are stable and can learn.")
#     print("----------------------------------------------")

#     # --- 1. Set up DataLoader ---
#     train_loader = DataLoader(
#         Stage2StackDataset(
#             simd_r_drive_server_config=simd_r_drive_server_config,
#             shuffle=True,
#             lookup_batch_size=64,
#         ),
#         batch_size=CONFIG["batch_size"],
#         collate_fn=stage2_collate_stacks,
#         num_workers=0, # Use 0 workers for simplicity in debugging
#     )

#     # --- 2. Create Model ---
#     model = SingleStackTester(CONFIG)

#     # --- 3. Set up Trainer ---
#     trainer = pl.Trainer(
#         max_epochs=5, # Run for a few epochs to see if loss moves
#         accelerator="auto",
#         devices=1,
#         enable_progress_bar=True,
#         enable_checkpointing=False,
#         logger=True, 
#         gradient_clip_val=CONFIG["grad_clip_val"],
#     )

#     # --- 4. Run the Test ---
#     try:
#         print("Starting trainer.fit()...")
#         trainer.fit(model, train_dataloaders=train_loader)
#         print("\n--- Diagnostic Finished ---")
#     except Exception as e:
#         print(f"\nFATAL: Training crashed during diagnostic. Error: {e}")

# if __name__ == "__main__":
#     run_diagnostic()


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from typing import List, Tuple, Dict
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR

# --- BUILDING BLOCK MODULES ---

class PerceiverStackEncoder(nn.Module):
    """
    The encoder for a single stack. Distills a variable-length input into a fixed-size latent representation.
    """
    def __init__(self, input_dim: int, latent_dim: int, num_latents: int, depth: int, dropout_rate: float):
        super().__init__()
        self.latents = nn.Parameter(torch.randn(num_latents, latent_dim))
        self.cross_attn = nn.MultiheadAttention(embed_dim=latent_dim, num_heads=4, batch_first=True, dropout=dropout_rate)
        self.cross_proj = nn.Linear(input_dim, latent_dim)
        
        self.query_norm = nn.LayerNorm(latent_dim)
        self.kv_norm = nn.LayerNorm(latent_dim) 

        self.blocks = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(
                    d_model=latent_dim, nhead=4, dim_feedforward=latent_dim * 4,
                    batch_first=True, activation=F.gelu, norm_first=True,
                    dropout=dropout_rate
                ) for _ in range(depth)
            ]
        )
        self.output_norm = nn.LayerNorm(latent_dim)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, N, latent_dim = x.shape[0], x.shape[1], self.latents.size(-1)
        
        latents = self.latents.unsqueeze(0).repeat(B, 1, 1)
        x_proj = self.cross_proj(x)

        latents_norm = self.query_norm(latents)
        x_proj_norm = self.kv_norm(x_proj)
        
        key_padding_mask = ~mask
        fully_padded_items = ~mask.any(dim=1)
        if fully_padded_items.any():
            key_padding_mask[fully_padded_items, 0] = False

        attn_output, _ = self.cross_attn(
            query=latents_norm, 
            key=x_proj_norm, 
            value=x_proj_norm, 
            key_padding_mask=key_padding_mask
        )
        
        latents = latents + attn_output
        for block in self.blocks:
            latents = block(latents)
            
        return self.output_norm(latents).mean(dim=1)


class PerceiverDecoder(nn.Module):
    """
    The decoder module. Reconstructs a stack using a shared context and a task-specific "anchor" vector.
    """
    def __init__(self, shared_dim: int, anchor_dim: int, query_dim: int, output_dim: int, depth: int, dropout_rate: float):
        super().__init__()
        
        # Projection layer for the combined context
        self.context_proj = nn.Linear(shared_dim + anchor_dim, query_dim)
        
        self.cross_attn = nn.MultiheadAttention(embed_dim=query_dim, num_heads=4, batch_first=True, dropout=dropout_rate)
        
        self.query_norm = nn.LayerNorm(query_dim)
        self.context_norm = nn.LayerNorm(query_dim) 

        self.blocks = nn.ModuleList(
            [
                nn.TransformerEncoderLayer(
                    d_model=query_dim, nhead=4, dim_feedforward=query_dim * 4,
                    batch_first=True, activation=F.gelu, norm_first=True,
                    dropout=dropout_rate
                ) for _ in range(depth)
            ]
        )
        self.output_head = nn.Linear(query_dim, output_dim)

    def forward(self, shared_vector: torch.Tensor, anchor_vector: torch.Tensor, output_query: torch.Tensor) -> torch.Tensor:
        # Combine shared and anchor vectors to form the context
        combined_context = torch.cat([shared_vector, anchor_vector], dim=-1)
        projected_context = self.context_proj(combined_context)
        
        latent = projected_context.unsqueeze(1).expand(-1, output_query.size(1), -1)

        query_norm = self.query_norm(output_query)
        latent_norm = self.context_norm(latent)

        attn_output, _ = self.cross_attn(
            query=query_norm, 
            key=latent_norm, 
            value=latent_norm
        )

        x = output_query + attn_output

        for block in self.blocks:
            x = block(x)
        return self.output_head(x)

# --- MAIN LIGHTNING MODULE (ANCHOR VECTOR ARCHITECTURE) ---

class Stage2Autoencoder(pl.LightningModule):
    def __init__(
        self,
        loss_alpha: float = 0.5,
        dropout_rate: float = 0.1,
        weight_decay: float = 1e-4,
        warmup_steps: int = 500,
        num_categories: int = 6,
        input_dim: int = 256,
        encoder_latent_dim: int = 128,
        shared_latent_dim: int = 512, # The dimension of the single vector
        num_latents: int = 64,
        encoder_depth: int = 2,
        query_dim: int = 256,
        decoder_depth: int = 2,
        lr: float = 1e-4,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.stats = {}
        
        self.encoders = nn.ModuleList(
            [
                PerceiverStackEncoder(input_dim, encoder_latent_dim, num_latents, encoder_depth, dropout_rate)
                for _ in range(num_categories)
            ]
        )
        self.encoder_to_shared = nn.Linear(num_categories * encoder_latent_dim, shared_latent_dim)

        self.balance_embedding = nn.Embedding(num_embeddings=3, embedding_dim=query_dim)
        self.period_embedding = nn.Embedding(num_embeddings=3, embedding_dim=query_dim)
        
        # The decoder now takes the shared dim and the anchor dim
        self.decoder = PerceiverDecoder(shared_latent_dim, encoder_latent_dim, query_dim, input_dim, decoder_depth, dropout_rate)
        self.loss_fn = nn.MSELoss()

    def on_train_epoch_start(self):
        self.stats['train'] = {'running_loss': 0.0, 'batches_seen': 0}

    def on_validation_epoch_start(self):
        self.stats['val'] = {'running_loss': 0.0, 'batches_seen': 0}

    def forward(
        self,
        stacks: List[torch.Tensor],
        masks: List[torch.Tensor],
        balance_batch: List[torch.Tensor],
        period_batch: List[torch.Tensor]
    ) -> Tuple[List[torch.Tensor], torch.Tensor]:
        
        # 1. Create the task-specific "anchor" vectors
        anchor_vectors = []
        batch_size = stacks[0].size(0) if stacks else 1
        for i in range(self.hparams.num_categories):
            if i < len(stacks):
                output = self.encoders[i](stacks[i], masks[i])
                anchor_vectors.append(output)
            else:
                zero_output = torch.zeros(batch_size, self.hparams.encoder_latent_dim, device=self.device)
                anchor_vectors.append(zero_output)
        
        # 2. Create the SINGLE shared latent vector from the anchors
        concatenated_anchors = torch.cat(anchor_vectors, dim=1)
        shared_latent = self.encoder_to_shared(concatenated_anchors)

        # 3. Decode each category using the shared vector AND its specific anchor
        reconstructions = []
        for i in range(len(balance_batch)):
            anchor_vector = anchor_vectors[i]
            
            balance_embed = self.balance_embedding(balance_batch[i])
            period_embed = self.period_embedding(period_batch[i])
            query = balance_embed + period_embed
            
            recon = self.decoder(shared_latent, anchor_vector, query)
            reconstructions.append(recon)
            
        # 4. Return both reconstructions and the single shared vector
        return reconstructions, shared_latent

    def _shared_step(self, batch: Tuple) -> Dict[str, torch.Tensor]:
        stacks, masks, balance_batch, period_batch = batch
        # The forward pass now returns the shared_latent as well
        reconstructions, shared_latent = self.forward(stacks, masks, balance_batch, period_batch)
        
        total_combined_loss = 0
        total_mse_loss = 0
        total_cosine_loss = 0
        num_valid_stacks = 0
        
        for i in range(len(reconstructions)):
            original = stacks[i][masks[i]]
            recon = reconstructions[i][masks[i]]
            if original.numel() > 0:
                mse_loss = self.loss_fn(recon, original)
                cosine_loss = (1 - torch.nn.functional.cosine_similarity(
                    recon, original, dim=1, eps=1e-8
                )).mean()
                combined_loss = self.hparams.loss_alpha * mse_loss + (1 - self.hparams.loss_alpha) * cosine_loss
                total_combined_loss += combined_loss
                total_mse_loss += mse_loss.detach()
                total_cosine_loss += cosine_loss.detach()
                num_valid_stacks += 1
        
        if num_valid_stacks == 0:
             return {'loss': torch.tensor(0.0, device=self.device, requires_grad=True), 'mse_loss': 0, 'cosine_loss': 0}

        avg_mse = total_mse_loss / num_valid_stacks
        avg_cosine = total_cosine_loss / num_valid_stacks
        return {
            'loss': total_combined_loss, 
            'mse_loss': avg_mse, 
            'cosine_loss': avg_cosine
        }

    def training_step(self, batch, batch_idx):
        losses = self._shared_step(batch)
        batch_total_loss = losses['loss']
        if not torch.isfinite(batch_total_loss):
            return None
        train_stats = self.stats['train']
        train_stats['running_loss'] += batch_total_loss.item()
        train_stats['batches_seen'] += 1
        running_avg_loss = train_stats['running_loss'] / train_stats['batches_seen']
        self.log("train_loss_running", running_avg_loss, on_step=True, on_epoch=False, prog_bar=True, logger=True)
        self.log("train_loss_epoch", batch_total_loss, on_step=False, on_epoch=True)
        self.log("train_mse_epoch", losses['mse_loss'], on_step=False, on_epoch=True)
        self.log("train_cosine_epoch", losses['cosine_loss'], on_step=False, on_epoch=True)
        return batch_total_loss

    def validation_step(self, batch, batch_idx):
        losses = self._shared_step(batch)
        if losses is None or not torch.isfinite(losses['loss']):
            return None
        val_stats = self.stats['val']
        val_stats['running_loss'] += losses['loss'].item()
        val_stats['batches_seen'] += 1
        self.log("val_mse_epoch", losses['mse_loss'], on_step=False, on_epoch=True)
        self.log("val_cosine_epoch", losses['cosine_loss'], on_step=False, on_epoch=True)
        return None

    def on_validation_epoch_end(self):
        val_stats = self.stats['val']
        if val_stats['batches_seen'] > 0:
            avg_val_loss = val_stats['running_loss'] / val_stats['batches_seen']
            self.log("val_loss_epoch", avg_val_loss, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), 
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay
        )
        try:
            total_steps = self.trainer.estimated_stepping_batches
        except AttributeError:
            total_steps = 100_000 
        warmup_scheduler = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=self.hparams.warmup_steps)
        main_scheduler_t_max = max(1, total_steps - self.hparams.warmup_steps)
        main_scheduler = CosineAnnealingLR(optimizer, T_max=main_scheduler_t_max, eta_min=1e-7)
        lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer, schedulers=[warmup_scheduler, main_scheduler], milestones=[self.hparams.warmup_steps]
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": lr_scheduler,
                "interval": "step",
            },
        }


In [ ]:
import os
import torch
import optuna
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import EarlyStopping
from torch.utils.data import DataLoader

# --- Make sure these imports are correct for your project structure ---
from config import project_paths, simd_r_drive_server_config
from models.pytorch.narrative_stack.stage2.dataset import (
    Stage2StackDataset,
    stage2_collate_stacks,
)

# ======== TUNING CONFIGURATION ========
OUTPUT_PATH = project_paths.python_data / "stage2_tuning"
os.makedirs(OUTPUT_PATH, exist_ok=True)
OPTUNA_DB_PATH = f"sqlite:///{OUTPUT_PATH / 'stage2_study.db'}"

# Training settings for each trial
EPOCHS = 5
PATIENCE = 5
N_TRIALS = 25
LOOKUP_BATCH_SIZE = 64
NUM_WORKERS = 4

# ======== OPTUNA OBJECTIVE FUNCTION ========

def objective(trial: optuna.trial.Trial) -> float:
    """
    This function defines a single training run (a "trial").
    Optuna will call this function multiple times with different hyperparameter values.
    """
    # --- 1. Suggest Hyperparameters ---
    lr = trial.suggest_float("lr", 1e-5, 5e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.05, 0.3, step=0.05)
    loss_alpha = trial.suggest_float("loss_alpha", 0.2, 0.8)
    warmup_steps = trial.suggest_categorical("warmup_steps", [250, 500, 1000])
    
    # Encoder architecture
    encoder_latent_dim = trial.suggest_categorical("encoder_latent_dim", [128, 192])
    num_latents = trial.suggest_categorical("num_latents", [64, 128])
    encoder_depth = trial.suggest_int("encoder_depth", 3, 6)

    # Decoder and shared architecture
    shared_latent_dim = trial.suggest_categorical("shared_latent_dim", [512, 768])
    query_dim = trial.suggest_categorical("query_dim", [192, 256])
    decoder_depth = trial.suggest_int("decoder_depth", 3, 6)

    # --- 2. Set up DataLoaders ---
    train_loader = DataLoader(
        Stage2StackDataset(
            simd_r_drive_server_config=simd_r_drive_server_config,
            shuffle=True,
            lookup_batch_size=LOOKUP_BATCH_SIZE,
        ),
        batch_size=batch_size,
        collate_fn=stage2_collate_stacks,
        num_workers=NUM_WORKERS,
        persistent_workers=True if NUM_WORKERS > 0 else False,
    )

    val_loader = DataLoader(
        Stage2StackDataset(
            simd_r_drive_server_config=simd_r_drive_server_config,
            shuffle=False,
            lookup_batch_size=LOOKUP_BATCH_SIZE,
        ),
        batch_size=batch_size,
        collate_fn=stage2_collate_stacks,
        num_workers=NUM_WORKERS,
        persistent_workers=True if NUM_WORKERS > 0 else False,
    )

    # --- 3. Create Model with Suggested Hyperparameters ---
    model = Stage2Autoencoder(
        lr=lr,
        loss_alpha=loss_alpha,
        dropout_rate=dropout_rate,
        weight_decay=weight_decay,
        warmup_steps=warmup_steps,
        encoder_latent_dim=encoder_latent_dim,
        num_latents=num_latents,
        encoder_depth=encoder_depth,
        shared_latent_dim=shared_latent_dim,
        query_dim=query_dim,
        decoder_depth=decoder_depth,
    )

    # --- 4. Set up Trainer ---
    logger = TensorBoardLogger(OUTPUT_PATH, name="tuning_logs", version=f"trial_{trial.number}")
    early_stop_callback = EarlyStopping(monitor="val_loss_epoch", patience=PATIENCE, verbose=True, mode="min")

    trainer = pl.Trainer(
        max_epochs=EPOCHS,
        logger=logger,
        callbacks=[early_stop_callback],
        accelerator="auto",
        devices=1,
        enable_progress_bar=True,
        enable_checkpointing=False,
        gradient_clip_val=1.0, # A fixed, sensible value is often sufficient

        # Limit train and val batches to 20%
        limit_train_batches=0.2, 
        limit_val_batches=0.2,
    )

    # --- 5. Train the Model ---
    try:
        trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
    except Exception as e:
        print(f"Trial {trial.number} failed with error: {e}")
        return float('inf')

    # --- 6. Return the Metric to Minimize ---
    val_loss = trainer.callback_metrics.get("val_loss_epoch", float('inf'))
    return val_loss

# ======== MAIN EXECUTION ========
if __name__ == "__main__":
    print(f"Starting Optuna study. Database at: {OPTUNA_DB_PATH}")
    study = optuna.create_study(
        direction="minimize",
        storage=OPTUNA_DB_PATH,
        study_name="stage2_autoencoder_tuning_v2", # Use a new name for the new study
        load_if_exists=True,
        pruner=optuna.pruners.MedianPruner()
    )
    study.optimize(objective, n_trials=N_TRIALS)

    print("\n\n--- TUNING STUDY COMPLETE ---")
    print(f"Number of finished trials: {len(study.trials)}")
    print(f"Best trial value: {study.best_trial.value:.6f}")
    print("Best Hyperparameters:")
    for key, value in study.best_trial.params.items():
        print(f"    {key}: {value}")


In [ ]:
# import os
# from pathlib import Path
# from pytorch_lightning.loggers import TensorBoardLogger
# from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
# from torch.utils.data import DataLoader
# import pytorch_lightning as pl

# from config import project_paths, simd_r_drive_server_config
# # from models.pytorch.narrative_stack.stage2.model import Stage2Autoencoder
# from models.pytorch.narrative_stack.stage2.dataset import (
#     Stage2StackDataset,
#     stage2_collate_stacks,
# )

# # === CONFIG ===
# OUTPUT_PATH = Path(project_paths.python_data / "stage2_training_output")
# os.makedirs(OUTPUT_PATH, exist_ok=True)

# EPOCHS = 1000
# PATIENCE = 20
# BATCH_SIZE = 32  # Safe to increase; collate handles padding
# NUM_WORKERS = 4
# LOOKUP_BATCH_SIZE = 64
# GRAD_CLIP = 0.5

# # === MODEL ===
# model = Stage2Autoencoder()

# # === DATALOADERS ===
# train_loader = DataLoader(
#     Stage2StackDataset(
#         simd_r_drive_server_config=simd_r_drive_server_config,
#         shuffle=True,
#         lookup_batch_size=LOOKUP_BATCH_SIZE,
#     ),
#     batch_size=BATCH_SIZE,
#     collate_fn=stage2_collate_stacks,
#     pin_memory=True,
#     persistent_workers=True,
#     prefetch_factor=4,
#     num_workers=NUM_WORKERS,
# )

# val_loader = DataLoader(
#     Stage2StackDataset(
#         simd_r_drive_server_config=simd_r_drive_server_config,
#         shuffle=False,
#         lookup_batch_size=LOOKUP_BATCH_SIZE,
#     ),
#     batch_size=BATCH_SIZE,
#     collate_fn=stage2_collate_stacks,
#     pin_memory=True,
#     persistent_workers=True,
#     prefetch_factor=4,
#     num_workers=NUM_WORKERS,
# )

# # === CALLBACKS ===
# early_stop_callback = EarlyStopping(
#     monitor="train_total_loss",
#     patience=PATIENCE,
#     verbose=True,
#     mode="min",
# )

# model_checkpoint = ModelCheckpoint(
#     dirpath=OUTPUT_PATH,
#     filename="stage2_checkpoint",
#     monitor="train_total_loss",
#     mode="min",
#     save_top_k=1,
#     verbose=True,
# )

# # === TRAINER ===
# trainer = pl.Trainer(
#     max_epochs=EPOCHS,
#     logger=TensorBoardLogger(OUTPUT_PATH, name="stage2_autoencoder"),
#     callbacks=[early_stop_callback, model_checkpoint],
#     accelerator="auto",
#     devices=1,
#     gradient_clip_val=GRAD_CLIP,
# )

# # === TRAIN ===
# trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)
